In [1]:
import sys
import os
import torch.nn.functional as F
project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.processor.word_process import sudachi_tokenize
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch

/home/manaty/company-classification-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/raw/industry_master_33_refined.csv")
print(df.shape)
df.head()

(33, 2)


,業種名,業種説明
0,水産・農林業,本業種は、農業、林業、および水産養殖業を含む漁業に関する事業を行う企業が分類される業種である...
1,鉱業,本業種は、天然に固体・液体・ガスの状態で存在する鉱物を掘採し、選鉱や品位向上処理を行う事業で...
2,建設業,本業種は、土木施設や建築物の建設工事を発注者から直接請け負う、あるいは自ら手掛ける事業を行う...
3,食料品,本業種は、畜産食料品、水産食料品、農産加工品、調味料、精穀・製粉、パン・菓子、めん類、豆腐、...
4,繊維製品,本業種は、製糸、紡績、織物、ニット生地、染色整理、縫製など、糸や生地から繊維製品を製造する事...


In [3]:
from src.processor.text_processor import chunk_text_by_sentence

chunked_data = []
for idx, row in df.iterrows():
    chunks = chunk_text_by_sentence(row["業種説明"])
    for chunk in chunks:
        chunked_data.append({
            "業種名": row["業種名"],
            "chunk_text": f"passage: {chunk}"
        })

df_chunks = pd.DataFrame(chunked_data)
print(f"全チャンク数: {len(df_chunks)}")
df_chunks.head()

全チャンク数: 33


,業種名,chunk_text
0,水産・農林業,passage: 本業種は、農業、林業、および水産養殖業を含む漁業に関する事業を行う企業が分...
1,鉱業,passage: 本業種は、天然に固体・液体・ガスの状態で存在する鉱物を掘採し、選鉱や品位向...
2,建設業,passage: 本業種は、土木施設や建築物の建設工事を発注者から直接請け負う、あるいは自ら...
3,食料品,passage: 本業種は、畜産食料品、水産食料品、農産加工品、調味料、精穀・製粉、パン・菓...
4,繊維製品,passage: 本業種は、製糸、紡績、織物、ニット生地、染色整理、縫製など、糸や生地から繊...


In [4]:
'''
#ファインチューニング前のSBERTによるベクトル化
#日本語対応のモデルをロード
model = SentenceTransformer('intfloat/multilingual-e5-base')
#企業の事業内容を一気にベクトル化(学習時間およそ10分)
sentences = df_chunks["chunk_text"].tolist()
category_embeddings = model.encode(sentences, show_progress_bar=True)
'''

'\n#ファインチューニング前のSBERTによるベクトル化\n#日本語対応のモデルをロード\nmodel = SentenceTransformer(\'intfloat/multilingual-e5-base\')\n#企業の事業内容を一気にベクトル化(学習時間およそ10分)\nsentences = df_chunks["chunk_text"].tolist()\ncategory_embeddings = model.encode(sentences, show_progress_bar=True)\n'

In [5]:
#チャンク化してベクトル化したものを業種ごとに平均をとる
print(category_embeddings.shape)#業種は一対一対応

df_chunks["embedding"] = list(category_embeddings)
grouped_series = df_chunks.groupby("業種名")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0))

df_category_embeddings = pd.DataFrame(
    grouped_series.tolist(),
    index=grouped_series.index)

final_embeddings = df_category_embeddings.values

print("企業名付きデータフレームの形状:", df_category_embeddings.shape) # (3637, 768)
print("NumPy配列の形状:", final_embeddings.shape)
df_category_embeddings.head()

NameError: name 'category_embeddings' is not defined

In [ ]:
"""
#次回のために保存
np.save("../models/neo_category_embeddings.npy", final_embeddings)

# 1. 企業名付きデータフレーム（33, 768）を pickle で保存する
df_category_embeddings.to_pickle('../models/neo_category_data_with_sbert.pkl')
"""

In [5]:
#ファインチューニング後のSBERTモデルによるベクトル化
# 1. 育てたモデルのパスを指定（Classification用ではなく、素のAutoModelで読み込む）
model_path = "../models/e5_edinet_finetuned"
print("育てたモデルを読み込み中...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModel.from_pretrained(model_path)


# 2. チャンクテキストのリストを取得
sentences = df_chunks["chunk_text"].tolist()

# 3. 平均プーリング（Mean Pooling）関数の定義
# E5/BERTの生出力を正しい768次元の文章ベクトルに変換するために必要です
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# 4. バッチ処理で一気にベクトル化
batch_size = 32  # メモリに応じて調整
all_embeddings = []

print("ファインチューニング済みモデルでベクトル化を実行中...")
for i in tqdm(range(0, len(sentences), batch_size)):
    batch_sentences = sentences[i:i + batch_size]
    
    # トークナイズ
    encoded_input = tokenizer(
        batch_sentences, 
        padding=True, 
        truncation=True, 
        max_length=512, 
        return_tensors='pt'
    )
    
    # ベクトル計算
    with torch.no_grad():
        model_output = base_model(**encoded_input)
        # トークンごとのベクトルから、文章全体の平均ベクトル（768次元）を抽出
        batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        # CPUに書き戻してリストに追加
        all_embeddings.append(batch_embeddings.cpu().numpy())

# 特徴ベクトルを1つのNumPy配列に結合
category_embeddings = np.vstack(all_embeddings)
print("ファインチューニングモデルによるベクトル化の形状:", category_embeddings.shape)

育てたモデルを読み込み中...


Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 1115.15it/s]
[transformers] XLMRobertaModel LOAD REPORT from: ../models/e5_edinet_finetuned
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ファインチューニング済みモデルでベクトル化を実行中...


100%|█████████████████████████████████████████████| 2/2 [00:02<00:00,  1.32s/it]

ファインチューニングモデルによるベクトル化の形状: (33, 768)


In [8]:
df_chunks["embedding"] = list(category_embeddings)
grouped_series = df_chunks.groupby("業種名")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0))

df_final_embeddings = pd.DataFrame(
    grouped_series.tolist(),
    index=grouped_series.index)

final_embeddings = df_final_embeddings.values

print("業種データフレームの形状:", df_final_embeddings.shape) # (3637, 768)
print("NumPy配列の形状:", final_embeddings.shape)
df_final_embeddings.head()

業種データフレームの形状: (33, 768)
NumPy配列の形状: (33, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
業種名,,,,,,,,,,,,,,,,,,,,,
その他製品,-0.002413,-0.229338,-0.125678,-0.180158,0.450537,-0.490937,-0.022474,0.043109,0.935214,0.335971,...,-0.021822,-0.150155,0.842878,-0.067125,-0.004864,-0.514601,0.239081,-0.342837,0.219459,0.911567
その他金融業,0.078164,0.893303,0.461719,0.277800,0.291229,-0.768364,0.077983,0.143596,-0.155588,-0.417994,...,-0.501690,0.246909,-0.272016,0.710290,0.252154,0.025707,0.176287,-0.000371,-0.194312,0.021282
ガラス・土石製品,0.118056,0.187316,-0.275980,-0.467367,-0.235589,-0.273879,-0.520113,-0.240110,0.543073,0.211649,...,-0.670757,-0.406785,0.118584,-1.169611,-0.636818,-0.617979,0.290294,0.537728,-0.277434,-0.498841
ゴム製品,0.400958,0.568359,-0.307757,-1.260276,0.242776,-0.017908,-0.068972,0.223536,1.030773,0.163748,...,-0.898011,-0.684903,-0.583152,-0.623568,-0.524528,-0.597851,-0.255985,-0.083769,0.167141,-0.899687
サービス業,0.044275,-0.106735,0.306515,0.117052,0.592460,-0.099549,-0.083255,-0.440345,-0.010550,-0.509826,...,0.283919,-0.424087,0.553469,0.117711,-0.047983,-0.219966,0.175200,-0.089082,-0.338762,-0.096113


In [9]:
#次回のために保存
np.save("../models/true_neo_category_embeddings.npy", category_embeddings)

# 1. 企業名付きデータフレーム（33, 768）を pickle で保存する
df_final_embeddings.to_pickle('../models/true_neo_category_data_with_sbert.pkl')